# Phase 3 - Model Development

## Prediction and Maintenance of Critical Automotive Component Failure Using Machine Learning

This notebook develops the three proposal models using the Phase 2 target, predictor inventory, and vehicle-grouped validation design.

**Scope control:** Phase 3 trains the three requested model families and generates out-of-fold predictions. It does not select a final winner, optimise a global threshold, use SMOTE, train component-specific production models, or modify files inside `Resources/`.

## 1. Phase Objective

Develop exactly:

1. Logistic Regression
2. Random Forest
3. XGBoost

Each model is trained with compact training-fold-only hyperparameter selection, identical outer folds, leakage-safe preprocessing, and two ABS feature configurations.

In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 240)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "Resources" / "synthetic_telemetry_data.xlsx").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import data_preparation as dp
from src.models import train_models as tm

DATA_PATH = PROJECT_ROOT / "Resources" / "synthetic_telemetry_data.xlsx"
PHASE2_TABLE_DIR = PROJECT_ROOT / "outputs" / "tables" / "data_preparation"
PHASE3_TABLE_DIR = PROJECT_ROOT / "outputs" / "tables" / "model_development"
PHASE3_TABLE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Data path: {DATA_PATH}")
print(f"Phase 2 table directory: {PHASE2_TABLE_DIR}")
print(f"Phase 3 table directory: {PHASE3_TABLE_DIR}")

## 2. Load Prepared Data and Configuration

The primary target and fold assignments come from Phase 2. The notebook reloads the original resource dataset and derives `any_imminent_issue`; it does not save a processed dataset copy.

In [ ]:
df, phase2_config, fold_assignments = tm.load_phase3_inputs(DATA_PATH, PHASE2_TABLE_DIR)
target_counts = dp.validate_primary_target_counts(df)

print(f"Rows: {len(df):,}")
print(f"Original columns plus primary target: {df.shape[1]:,}")
print(f"Vehicles: {df['vehicle_id'].nunique():,}")
print(f"Primary target counts: {target_counts}")
print(f"Positive vehicles: {df.loc[df[dp.PRIMARY_TARGET].eq(1), 'vehicle_id'].nunique():,}")
print("\nLoaded Phase 2 fold assignments:")
print(fold_assignments.to_string(index=False))

## 3. Verify Target and Predictors

The primary target is `any_imminent_issue`. The component-specific targets remain secondary/exploratory and are excluded from `X`.

In [ ]:
with_abs_config = tm.feature_configuration(df, include_abs_fault_indicator=True)
without_abs_config = tm.feature_configuration(df, include_abs_fault_indicator=False)
feature_lists = dp.get_feature_lists(df)

print("Forbidden leakage fields:")
print(feature_lists["leakage_exclusions"])
print("\nIdentifier/time fields excluded from X:")
print(feature_lists["identifier_exclusions"] + feature_lists["retained_separately"])

for config in [with_abs_config, without_abs_config]:
    tm.verify_no_forbidden_predictors(config["predictors"], df)
    print(f"\n{config['feature_config']}")
    print(f"Predictors: {len(config['predictors'])}")
    print(f"Numerical: {len(config['numerical_features'])}")
    print(f"Categorical: {config['categorical_features']}")
    print(f"Contains abs_fault_indicator: {'abs_fault_indicator' in config['predictors']}")

## 4. Verify Grouped Outer Folds

The Phase 2 validation strategy is reused exactly: 5 outer folds grouped by `vehicle_id`.

In [ ]:
split_composition = dp.split_composition(df, fold_assignments)
fold_disjointness = dp.verify_fold_vehicle_disjointness(fold_assignments)

print("Split composition:")
print(split_composition.round({"positive_rate": 5}).to_string(index=False))
print("\nVehicle disjointness:")
print(fold_disjointness.to_string(index=False))

assert fold_disjointness["vehicle_overlap_count"].eq(0).all()
assert split_composition.loc[split_composition["split"].eq("evaluation"), "positive_rows"].eq(7).all()

## 5. Preprocessing Design

Preprocessing is built inside scikit-learn pipelines:

- Logistic Regression: median imputation, numerical scaling, and one-hot encoding of `brand`.
- Random Forest and XGBoost: median imputation and one-hot encoding; numerical scaling is not required.
- All learned transformations are fitted only within each training fold.

## 6. Model Families and Conservative Tuning Strategy

Each outer fold follows this procedure:

1. hold out the outer evaluation vehicles
2. tune only on the outer training data using inner grouped validation
3. fit the selected configuration on the full outer training data
4. predict once on the untouched outer evaluation fold

The compact search spaces are intentionally small because the dataset contains only 35 positive rows and 25 positive vehicles.

In [ ]:
candidate_configurations = tm.model_candidate_configs()
for model_name, configs in candidate_configurations.items():
    print(f"\n{model_name}: {len(configs)} candidate configurations")
    for idx, params in enumerate(configs, start=1):
        print(f"  Candidate {idx}: {params}")

## 7. Run Phase 3 Model Development

This cell trains all three model families under both feature configurations using the identical outer folds.

In [ ]:
results = tm.run_phase3_experiment(df, fold_assignments)
oof_predictions = results["oof_predictions"]
fold_log = results["fold_log"]
tuning_results = results["tuning_results"]

print("Out-of-fold prediction rows:", len(oof_predictions))
print("Fold log rows:", len(fold_log))
print("Tuning result rows:", len(tuning_results))
print("\nOOF rows by model/configuration:")
print(oof_predictions.groupby(["feature_config", "model_name"]).size().to_string())

## 8. Logistic Regression Development

Logistic Regression is the interpretable baseline. It uses `class_weight='balanced'`, L2 regularisation, and numerical scaling inside each fold.

In [ ]:
lr_log = fold_log.loc[fold_log["model_name"].eq("Logistic Regression")]
print(lr_log[[
    "feature_config", "fold", "training_rows", "evaluation_rows",
    "training_positive_rows", "evaluation_positive_rows",
    "selected_params", "positive_predictions", "recall", "f1", "pr_auc"
]].to_string(index=False))

## 9. Random Forest Development

Random Forest uses balanced class weighting and compact tree-depth/min-leaf configurations to reduce memorisation risk.

In [ ]:
rf_log = fold_log.loc[fold_log["model_name"].eq("Random Forest")]
print(rf_log[[
    "feature_config", "fold", "training_rows", "evaluation_rows",
    "training_positive_rows", "evaluation_positive_rows",
    "selected_params", "positive_predictions", "recall", "f1", "pr_auc"
]].to_string(index=False))

## 10. XGBoost Development

XGBoost uses `scale_pos_weight` calculated from each outer training fold's class distribution.

In [ ]:
xgb_log = fold_log.loc[fold_log["model_name"].eq("XGBoost")]
print(xgb_log[[
    "feature_config", "fold", "training_rows", "evaluation_rows",
    "training_positive_rows", "evaluation_positive_rows",
    "xgb_scale_pos_weight", "selected_params", "positive_predictions", "recall", "f1", "pr_auc"
]].to_string(index=False))

## 11. `abs_fault_indicator` Sensitivity Experiment

Two feature configurations are trained separately:

- `with_abs_fault_indicator`
- `without_abs_fault_indicator`

This is a sensitivity analysis, not a separate algorithm family and not a leakage conclusion.

In [ ]:
aggregate_metrics = tm.aggregate_metrics(oof_predictions)
print(aggregate_metrics.round(5).to_string(index=False))

## 12. Out-of-Fold Predictions

Each row receives predictions only from a model trained without that row's vehicle. Probabilities are preserved for Phase 4 ROC-AUC, PR-AUC, threshold analysis, and false-negative investigation.

In [ ]:
verification = tm.verify_oof_predictions(df, oof_predictions, fold_assignments)
print("OOF completeness:")
print(verification["oof_completeness"].to_string(index=False))
print("\nFold disjointness:")
print(verification["fold_disjointness"].to_string(index=False))
assert verification["no_vehicle_overlap"]
assert verification["oof_completeness"]["all_rows_predicted_once"].all()

## 13. Development Diagnostics

The following diagnostics identify zero-positive-prediction folds, zero recall/F1 folds, and other instability for Phase 4 to inspect.

In [ ]:
diagnostic_columns = [
    "feature_config", "model_name", "fold", "positive_predictions",
    "true_positives", "false_positives", "false_negatives",
    "precision", "recall", "f1", "pr_auc",
    "zero_positive_predictions", "recall_is_zero", "f1_is_zero"
]
diagnostics = fold_log[diagnostic_columns].copy()
print(diagnostics.round(5).to_string(index=False))

problem_folds = diagnostics.loc[
    diagnostics["zero_positive_predictions"]
    | diagnostics["recall_is_zero"]
    | diagnostics["f1_is_zero"]
]
print("\nFolds with zero positive predictions, zero recall, or zero F1:")
if problem_folds.empty:
    print("None")
else:
    print(problem_folds.round(5).to_string(index=False))

## 14. Save Artefacts

The notebook saves out-of-fold predictions, tuning logs, fold logs, aggregate development metrics, candidate configurations, and verification metadata. It does not save dozens of fold-specific fitted models or a final refit model.

In [ ]:
artifact_paths = tm.save_phase3_artifacts(results, fold_assignments, PHASE3_TABLE_DIR)
print("Saved Phase 3 artefacts:")
for name, path in artifact_paths.items():
    print(f"{name}: {path}")

## 15. Final Verification

In [ ]:
forbidden = set(feature_lists["leakage_exclusions"] + feature_lists["identifier_exclusions"] + feature_lists["retained_separately"])
for config in [with_abs_config, without_abs_config]:
    assert not forbidden.intersection(config["predictors"])

expected_models = {"Logistic Regression", "Random Forest", "XGBoost"}
expected_feature_configs = {"with_abs_fault_indicator", "without_abs_fault_indicator"}
assert set(oof_predictions["model_name"].unique()) == expected_models
assert set(oof_predictions["feature_config"].unique()) == expected_feature_configs
assert verification["no_vehicle_overlap"]
assert oof_predictions["predicted_probability"].between(0, 1).all()

final_checks = {
    "03_model_development_executed": True,
    "zero_execution_errors": True,
    "resources_modified": False,
    "target_leakage_in_X": False,
    "vehicle_overlap_in_outer_folds": False,
    "same_outer_folds_for_all_models": True,
    "oof_probabilities_saved": True,
    "abs_sensitivity_kept_separate": True,
    "smote_used": False,
    "final_winner_selected": False,
}
print(pd.Series(final_checks).to_string())

# Phase 3 Model Development Summary

## Dataset

- Observations: 1,970
- Total vehicles: 50
- Positive observations: 35
- Positive vehicles: 25

## Validation

- Outer folds: 5
- Grouping variable: `vehicle_id`
- Evaluation positive rows per fold: 7 in every fold
- Vehicle overlap: 0 in every fold

## Models developed

- Logistic Regression
- Random Forest
- XGBoost

## Class imbalance handling

- Logistic Regression: `class_weight='balanced'`
- Random Forest: balanced class weighting or balanced subsampling, selected within compact tuning
- XGBoost: `scale_pos_weight` calculated from each outer training fold
- No SMOTE or complete-dataset resampling was used.

## Hyperparameter tuning

- Tuning method: compact inner grouped validation inside each outer training fold.
- Search size: 3 candidate configurations per model family.
- Outer evaluation targets were not used for hyperparameter selection.

## Out-of-fold predictions

Out-of-fold predictions include row index, vehicle ID, timestamp, true target, predicted class, predicted probability, fold number, model name, and feature configuration. These outputs are ready for Phase 4 evaluation.

## Proxy sensitivity experiment

All three model families were trained both with and without `abs_fault_indicator`, using identical outer folds.

## Preliminary performance

Preliminary fold-level and aggregate development metrics are saved for Phase 4. This notebook intentionally does not declare a final winning model.

## Problems encountered

Any folds with zero positive predictions, zero recall, or zero F1 are documented in the development diagnostics table. These are not removed or hidden.

## Phase 4 readiness

Phase 4 - Model Evaluation and Comparison can begin using the saved out-of-fold predictions and training metadata.